# EmoText MZ — Avaliação Experimental (Capítulo 5)

**PFC:** Ivany Mazia Massinga — LECC51/5976 — ISUTC

Este notebook executa, com dados reais e de forma reprodutível, todas as experiências do Capítulo 5:

| Exp. | Descrição | Dados |
|---|---|---|
| E0 | Baseline de classe maioritária | AfriSenti (PT-MZ, XI) e AfriEmo (PT-MZ) |
| E1 | Baseline léxico NRC EmoLex (português) + Kappa de Cohen com a anotação humana | AfriSenti PT-MZ, AfriEmo PT-MZ |
| E2 | Baseline TF-IDF + Regressão Logística | AfriSenti (PT-MZ, XI) e AfriEmo (PT-MZ) |
| E3 | Ajuste fino de mBERT, XLM-R e AfroXLMR (3 sementes) | AfriSenti (PT-MZ, XI) e AfriEmo (PT-MZ) |
| E4 | Zero-shot interlinguístico PT-MZ → Xitsonga | AfriSenti XI (teste) |
| E5 | Desempenho por categoria linguística (Português / Código-Misto / Xichangana) | Testes do AfriSenti |
| E6 | Matrizes de confusão, F1 por emoção, tabelas e material para os anexos | — |

## Como executar
1. Copie a pasta **`EmoText`** para a raiz do seu **Google Drive** (`O meu disco/EmoText`).
2. Abra este notebook no Colab a partir dessa pasta.
3. Menu **Ambiente de execução → Alterar tipo de ambiente de execução → GPU T4**.
4. Menu **Ambiente de execução → Executar tudo** e autorize o acesso ao Google Drive.
5. Tempo estimado com GPU T4: **50 a 70 minutos**. Os resultados são gravados no Drive à medida que cada modelo termina;
   se a sessão cair, basta voltar a **Executar tudo** — as execuções já concluídas são saltadas.
6. No fim, é descarregado o ficheiro **`resultados_EmoText.zip`** (também fica gravado em `EmoText/`).

## 1. Configuração

In [ ]:
import os
import sys
import subprocess

EM_COLAB = 'google.colab' in sys.modules
TESTE_RAPIDO = os.environ.get('EMOTEXT_TESTE_RAPIDO', '0') == '1'

if EM_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.6', 'accelerate>=1.0', 'langdetect==1.0.9'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_BASE = '/content/drive/MyDrive/EmoText'
    if not os.path.isdir(PASTA_BASE):
        candidatos = [os.path.join(r, d) for r, ds, _ in os.walk('/content/drive/MyDrive') for d in ds if d == 'EmoText']
        if not candidatos:
            raise FileNotFoundError('Pasta EmoText não encontrada no Google Drive. Copie-a para "O meu disco/EmoText".')
        PASTA_BASE = candidatos[0]
    PASTA_TEMP = '/content/tmp_treino'
else:
    PASTA_BASE = os.environ.get('EMOTEXT_BASE', os.getcwd())
    PASTA_TEMP = os.path.join(PASTA_BASE, '_tmp_treino')

PASTA_DADOS = os.path.join(PASTA_BASE, 'dados')
PASTA_RES = os.path.join(PASTA_BASE, 'resultados_teste' if TESTE_RAPIDO else 'resultados')
SUB = {k: os.path.join(PASTA_RES, k) for k in ['logs', 'execucoes', 'previsoes', 'relatorios', 'tabelas', 'figuras', 'anexos']}
for p in list(SUB.values()) + [PASTA_TEMP]:
    os.makedirs(p, exist_ok=True)
print('Pasta base:', PASTA_BASE)
print('Resultados:', PASTA_RES)

In [ ]:
import ast
import glob
import io
import json
import logging
import random
import re
import shutil
import time
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import transformers
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline, make_union
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score, cohen_kappa_score,
                             classification_report, confusion_matrix, multilabel_confusion_matrix)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments, TrainerCallback, set_seed)

SEMENTES = [42, 43, 44]
MODELOS = {
    'mBERT': 'google-bert/bert-base-multilingual-cased',
    'XLM-R': 'FacebookAI/xlm-roberta-base',
    'AfroXLMR': 'Davlan/afro-xlmr-base',
}
EXECUTAR_AFROXLMR_76L = True
MODELO_76L = ('AfroXLMR-76L', 'Davlan/afro-xlmr-large-76L')

HIPER = {
    'max_comprimento': 128,
    'taxa_aprendizagem': 2e-5,
    'taxa_aprendizagem_large': 1e-5,
    'tamanho_lote': 16,
    'decaimento_pesos': 0.01,
    'proporcao_aquecimento': 0.1,
    'epocas': {'senti_por': 3, 'senti_tso': 5, 'emo_ptmz': 5},
    'limiar_multirrotulo': 0.5,
    'selecao_modelo': 'melhor época por Macro-F1 na validação',
}
SENTIMENTOS = ['negative', 'neutral', 'positive']
SENT_PT = {'negative': 'Negativo', 'neutral': 'Neutro', 'positive': 'Positivo'}
EMOCOES = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
EMO_PT = {'anger': 'Raiva', 'disgust': 'Nojo', 'fear': 'Medo', 'joy': 'Alegria', 'sadness': 'Tristeza', 'surprise': 'Surpresa'}
TAREFAS = {
    'senti_por': 'Sentimento — AfriSenti Português MZ',
    'senti_tso': 'Sentimento — AfriSenti Xitsonga',
    'emo_ptmz': 'Emoções — AfriEmo Português MZ',
    'zeroshot_por_tso': 'Zero-shot PT-MZ → Xitsonga',
}
URL_NRC = 'http://saifmohammad.com/WebDocs/Lexicons/NRC-Emotion-Lexicon.zip'

if TESTE_RAPIDO:
    SEMENTES = [42, 43]
    MODELOS = {'XLM-R': 'FacebookAI/xlm-roberta-base'}
    EXECUTAR_AFROXLMR_76L = False
    HIPER['epocas'] = {'senti_por': 1, 'senti_tso': 1, 'emo_ptmz': 1}

## 2. Registo de execução (log) e ambiente

In [ ]:
CAMINHO_LOG = os.path.join(SUB['logs'], f'execucao_{datetime.now():%Y%m%d_%H%M%S}.log')
log = logging.getLogger('emotext')
log.setLevel(logging.INFO)
log.handlers.clear()
for h in [logging.FileHandler(CAMINHO_LOG, encoding='utf-8'), logging.StreamHandler(sys.stdout)]:
    h.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s', '%Y-%m-%d %H:%M:%S'))
    log.addHandler(h)
for nome_ruidoso in ['transformers', 'datasets', 'urllib3', 'filelock']:
    logging.getLogger(nome_ruidoso).setLevel(logging.ERROR)
transformers.utils.logging.set_verbosity_error()

DISPOSITIVO = 'cuda' if torch.cuda.is_available() else 'cpu'
ambiente = {
    'data_execucao': f'{datetime.now():%Y-%m-%d %H:%M:%S}',
    'plataforma': 'Google Colab' if EM_COLAB else 'Local',
    'gpu': torch.cuda.get_device_name(0) if DISPOSITIVO == 'cuda' else 'sem GPU (CPU)',
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'scikit-learn': sklearn.__version__,
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'matplotlib': matplotlib.__version__,
    'seaborn': sns.__version__,
}
try:
    import langdetect
    ambiente['langdetect'] = getattr(langdetect, '__version__', '1.0.9')
except ImportError:
    ambiente['langdetect'] = 'não instalado'
with open(os.path.join(SUB['anexos'], 'ambiente_execucao.json'), 'w', encoding='utf-8') as f:
    json.dump(ambiente, f, ensure_ascii=False, indent=2)
log.info('=' * 70)
log.info('EmoText MZ — Avaliação experimental do Capítulo 5')
for k, v in ambiente.items():
    log.info(f'  {k}: {v}')
if not EM_COLAB and DISPOSITIVO == 'cpu' and not TESTE_RAPIDO:
    log.warning('Sem GPU: o ajuste fino completo pode demorar várias horas.')

## 3. Dados
- **AfriSenti** (Muhammad et al. 2023): ficheiros oficiais `train/dev/test.tsv` do repositório da tarefa SemEval-2023 Task 12.
- **AfriEmo / BRIGHTER** (Muhammad et al. 2025): subconjunto `ptmz`; os ficheiros `dev` e `test` descarregados repetem cada texto
  (faixas A e C), pelo que são deduplicados por texto, recuperando os tamanhos oficiais (1.546 / 257 / 776).
- Todos os textos passam pela mesma limpeza do módulo `cleaner.py` (URLs, menções, HTML e caracteres de controlo).

In [ ]:
def limpar_texto(texto):
    if not isinstance(texto, str):
        return ''
    texto = re.sub(r'http\S+|www\.\S+', '', texto)
    texto = re.sub(r'@\w+', '', texto)
    texto = re.sub(r'<[^>]+>', '', texto)
    texto = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', texto)
    return re.sub(r'\s+', ' ', texto).strip()


def carregar_afrisenti(lingua):
    partes = {}
    for split in ['train', 'dev', 'test']:
        df = pd.read_csv(os.path.join(PASTA_DADOS, f'afrisenti_{lingua}_{split}.tsv'), sep='\t', dtype=str)
        df = df.rename(columns={'tweet': 'texto_original'})
        df['texto'] = df['texto_original'].apply(limpar_texto)
        df['y'] = df['label'].map({s: i for i, s in enumerate(SENTIMENTOS)})
        assert df['y'].notna().all(), f'Rótulo desconhecido em afrisenti_{lingua}_{split}'
        partes[split] = df.reset_index(drop=True)
    return partes


def carregar_afriemo():
    partes = {}
    for split in ['train', 'dev', 'test']:
        df = pd.read_csv(os.path.join(PASTA_DADOS, f'afriemo_ptmz_{split}.csv'), dtype={'text': str})
        df = df.drop_duplicates(subset='text').rename(columns={'text': 'texto_original'})
        df['texto'] = df['texto_original'].apply(limpar_texto)
        df[EMOCOES] = df[EMOCOES].astype(int)
        partes[split] = df.reset_index(drop=True)
    return partes


DADOS = {'senti_por': carregar_afrisenti('por'), 'senti_tso': carregar_afrisenti('tso'), 'emo_ptmz': carregar_afriemo()}

if TESTE_RAPIDO:
    for t in DADOS:
        DADOS[t] = {s: d.sample(min(len(d), 48 if s == 'train' else 24), random_state=0).reset_index(drop=True)
                    for s, d in DADOS[t].items()}

linhas = []
for t, partes in DADOS.items():
    for s, df in partes.items():
        dist = (df['label'].value_counts().to_dict() if t.startswith('senti')
                else {EMO_PT[e]: int(df[e].sum()) for e in EMOCOES} | {'Neutro': int((df[EMOCOES].sum(axis=1) == 0).sum())})
        linhas.append({'tarefa': TAREFAS[t], 'particao': s, 'n': len(df), 'distribuicao': json.dumps(dist, ensure_ascii=False)})
        log.info(f'[Dados] {t:10s} {s:5s} n={len(df):5d} {dist}')
pd.DataFrame(linhas).to_csv(os.path.join(SUB['tabelas'], 'T_dados_particoes.csv'), index=False, encoding='utf-8-sig')

## 4. Funções de avaliação e registo

In [ ]:
def metricas_sentimento(y_true, y_pred):
    return {
        'macro_f1': f1_score(y_true, y_pred, average='macro', labels=[0, 1, 2], zero_division=0),
        'f1_ponderado': f1_score(y_true, y_pred, average='weighted', labels=[0, 1, 2], zero_division=0),
        'exactidao': accuracy_score(y_true, y_pred),
        'kappa_cohen': cohen_kappa_score(y_true, y_pred, labels=[0, 1, 2]),
    }


def metricas_emocoes(Y_true, Y_pred):
    m = {
        'macro_f1': f1_score(Y_true, Y_pred, average='macro', zero_division=0),
        'micro_f1': f1_score(Y_true, Y_pred, average='micro', zero_division=0),
        'exactidao_exacta': accuracy_score(Y_true, Y_pred),
    }
    for i, e in enumerate(EMOCOES):
        m[f'f1_{e}'] = f1_score(Y_true[:, i], Y_pred[:, i], zero_division=0)
    return m


BASELINES = ('Maioritária', 'Léxico NRC', 'TF-IDF+RL')


def semente_de(modelo):
    return 0 if modelo in BASELINES else SEMENTES[0]


def chave_execucao(tarefa, modelo, semente):
    return f'{tarefa}__{modelo}__s{semente}'


def caminho_execucao(chave):
    return os.path.join(SUB['execucoes'], chave + '.json')


def ja_executado(chave):
    return os.path.exists(caminho_execucao(chave))


def registar_execucao(tarefa, modelo, semente, metricas, df_teste, previsto, segundos=0.0, extra=None):
    chave = chave_execucao(tarefa, modelo, semente)
    reg = {'tarefa': tarefa, 'modelo': modelo, 'semente': semente, 'segundos_treino': round(segundos, 1), **metricas, **(extra or {})}
    with open(caminho_execucao(chave), 'w', encoding='utf-8') as f:
        json.dump(reg, f, ensure_ascii=False, indent=2)
    prev = df_teste[['texto_original', 'texto']].copy()
    if tarefa.startswith('emo'):
        for i, e in enumerate(EMOCOES):
            prev[f'real_{e}'] = df_teste[e].values
            prev[f'prev_{e}'] = previsto[:, i]
        rel = classification_report(df_teste[EMOCOES].values, previsto, target_names=[EMO_PT[e] for e in EMOCOES], zero_division=0, digits=4)
    else:
        prev['real'] = [SENTIMENTOS[i] for i in df_teste['y']]
        prev['previsto'] = [SENTIMENTOS[i] for i in previsto]
        rel = classification_report(df_teste['y'], previsto, labels=[0, 1, 2], target_names=[SENT_PT[s] for s in SENTIMENTOS], zero_division=0, digits=4)
    prev.to_csv(os.path.join(SUB['previsoes'], chave + '.csv'), index=False, encoding='utf-8-sig')
    with open(os.path.join(SUB['relatorios'], chave + '.txt'), 'w', encoding='utf-8') as f:
        f.write(f'{TAREFAS[tarefa]} | modelo: {modelo} | semente: {semente}\n\n{rel}')
    resumo = ' | '.join(f'{k}={v:.4f}' for k, v in metricas.items() if k in ('macro_f1', 'f1_ponderado', 'exactidao', 'kappa_cohen', 'micro_f1'))
    log.info(f'[Resultado] {chave}: {resumo}')

## 5. E0 e E2 — Baselines não neuronais (classe maioritária e TF-IDF + Regressão Logística)

In [ ]:
def tfidf():
    return make_union(TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                      TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=2, sublinear_tf=True))


for tarefa in ['senti_por', 'senti_tso']:
    tr, te = DADOS[tarefa]['train'], DADOS[tarefa]['test']
    if not ja_executado(chave_execucao(tarefa, 'Maioritária', 0)):
        classe = int(tr['y'].mode()[0])
        prev = np.full(len(te), classe)
        registar_execucao(tarefa, 'Maioritária', 0, metricas_sentimento(te['y'], prev), te, prev)
    if not ja_executado(chave_execucao(tarefa, 'TF-IDF+RL', 0)):
        t0 = time.time()
        clf = make_pipeline(tfidf(), LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42))
        clf.fit(tr['texto'], tr['y'])
        prev = clf.predict(te['texto'])
        registar_execucao(tarefa, 'TF-IDF+RL', 0, metricas_sentimento(te['y'], prev), te, prev, time.time() - t0)

tr, te = DADOS['emo_ptmz']['train'], DADOS['emo_ptmz']['test']
if not ja_executado(chave_execucao('emo_ptmz', 'Maioritária', 0)):
    prev = np.tile((tr[EMOCOES].mean().values > 0.5).astype(int), (len(te), 1))
    registar_execucao('emo_ptmz', 'Maioritária', 0, metricas_emocoes(te[EMOCOES].values, prev), te, prev)
if not ja_executado(chave_execucao('emo_ptmz', 'TF-IDF+RL', 0)):
    t0 = time.time()
    clf = make_pipeline(tfidf(), OneVsRestClassifier(LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42)))
    clf.fit(tr['texto'], tr[EMOCOES].values)
    prev = clf.predict(te['texto'])
    registar_execucao('emo_ptmz', 'TF-IDF+RL', 0, metricas_emocoes(te[EMOCOES].values, prev), te, prev, time.time() - t0)

## 6. E1 — Baseline léxico NRC EmoLex (português) e Kappa de Cohen
O léxico NRC (Mohammad e Turney 2013) é descarregado da página oficial no momento da execução; os termos de utilização
permitem o uso para investigação, mas não a redistribuição, pelo que o léxico **não** é incluído nos anexos.
Não existe léxico de sentimento publicado para o xitsonga; o léxico português é aplicado ao xitsonga apenas para
documentar a limitação de recursos genéricos.

In [ ]:
def carregar_nrc_pt():
    destino = os.path.join(PASTA_TEMP, 'nrc.zip')
    if not os.path.exists(destino):
        import urllib.request
        pedido = urllib.request.Request(URL_NRC, headers={'User-Agent': 'curl/8.4.0'})
        with urllib.request.urlopen(pedido, timeout=120) as resposta, open(destino, 'wb') as f:
            shutil.copyfileobj(resposta, f)
    with zipfile.ZipFile(destino) as z:
        nome = [n for n in z.namelist() if n.endswith('Portuguese-NRC-EmoLex.txt')][0]
        df = pd.read_csv(io.BytesIO(z.read(nome)), sep='\t')
    df['termo'] = df['Portuguese Word'].astype(str).str.lower().str.strip()
    df = df[df['termo'].str.fullmatch(r"[\wÀ-ÿ'-]+")]
    cols = ['positive', 'negative'] + EMOCOES
    return df.groupby('termo')[cols].max()


def tokens(texto):
    return re.findall(r"[\wÀ-ÿ']+", texto.lower())


def nrc_sentimento(texto, lex):
    pos = neg = 0
    for t in tokens(texto):
        if t in lex.index:
            pos += lex.at[t, 'positive']
            neg += lex.at[t, 'negative']
    return 2 if pos > neg else 0 if neg > pos else 1


def nrc_emocoes(texto, lex):
    v = np.zeros(len(EMOCOES), dtype=int)
    for t in tokens(texto):
        if t in lex.index:
            v |= lex.loc[t, EMOCOES].values.astype(int)
    return v


NRC = carregar_nrc_pt()
log.info(f'[NRC] Léxico português carregado: {len(NRC)} termos únicos')
cobertura = {}
for tarefa in ['senti_por', 'senti_tso', 'emo_ptmz']:
    te = DADOS[tarefa]['test']
    toks = [t for x in te['texto'] for t in tokens(x)]
    cobertura[tarefa] = sum(t in NRC.index for t in toks) / max(len(toks), 1)
    if not ja_executado(chave_execucao(tarefa, 'Léxico NRC', 0)):
        if tarefa.startswith('senti'):
            prev = np.array([nrc_sentimento(x, NRC) for x in te['texto']])
            m = metricas_sentimento(te['y'], prev)
        else:
            prev = np.vstack([nrc_emocoes(x, NRC) for x in te['texto']])
            m = metricas_emocoes(te[EMOCOES].values, prev)
        registar_execucao(tarefa, 'Léxico NRC', 0, m, te, prev, extra={'cobertura_lexico': cobertura[tarefa]})

linhas = []
for tarefa, rotulo in [('senti_por', 'AfriSenti Português MZ (léxico em português)'),
                       ('senti_tso', 'AfriSenti Xitsonga (léxico português aplicado ao xitsonga)')]:
    r = json.load(open(caminho_execucao(chave_execucao(tarefa, 'Léxico NRC', 0)), encoding='utf-8'))
    prev = pd.read_csv(os.path.join(SUB['previsoes'], chave_execucao(tarefa, 'Léxico NRC', 0) + '.csv'))
    linhas.append({'conjunto': rotulo, 'n_teste': len(prev), 'cobertura_lexico_tokens': round(cobertura[tarefa], 4),
                   'kappa_cohen': round(r['kappa_cohen'], 4), 'exactidao': round(r['exactidao'], 4),
                   'macro_f1': round(r['macro_f1'], 4),
                   'previstos_neutro': int((prev['previsto'] == 'neutral').sum())})
pd.DataFrame(linhas).to_csv(os.path.join(SUB['tabelas'], 'T_kappa_lexico.csv'), index=False, encoding='utf-8-sig')
log.info('[Kappa] ' + json.dumps(linhas, ensure_ascii=False))

## 7. E3 e E4 — Ajuste fino dos modelos Transformer (3 sementes) e zero-shot PT-MZ → Xitsonga

In [ ]:
class RegistoTreino(TrainerCallback):

    def __init__(self, chave):
        self.chave = chave

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            itens = ' '.join(f'{k}={v:.4f}' if isinstance(v, float) else f'{k}={v}' for k, v in logs.items())
            log.info(f'    [{self.chave}] época={state.epoch:.2f} passo={state.global_step} {itens}')


class ConjuntoTexto(torch.utils.data.Dataset):
    def __init__(self, codificacoes, rotulos):
        self.cod, self.rot = codificacoes, rotulos

    def __len__(self):
        return len(self.rot)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.cod.items()}
        item['labels'] = self.rot[i]
        return item


def preparar(tok, df, multirrotulo):
    cod = tok(df['texto'].tolist(), truncation=True, max_length=HIPER['max_comprimento'])
    rot = [list(map(float, r)) for r in df[EMOCOES].values] if multirrotulo else df['y'].astype(int).tolist()
    return ConjuntoTexto(cod, rot)


def prever(trainer, conjunto, multirrotulo):
    logits = trainer.predict(conjunto).predictions
    if multirrotulo:
        return (1 / (1 + np.exp(-logits)) > HIPER['limiar_multirrotulo']).astype(int)
    return logits.argmax(axis=1)


def ajustar(tarefa, nome_modelo, id_modelo, semente, taxa):
    chave = chave_execucao(tarefa, nome_modelo, semente)
    if ja_executado(chave):
        log.info(f'[Saltar] {chave} já concluído')
        return
    multirrotulo = tarefa.startswith('emo')
    partes = DADOS[tarefa]
    log.info(f'[Treino] {chave} | {id_modelo} | lr={taxa} | épocas={HIPER["epocas"][tarefa]}')
    set_seed(semente)
    tok = AutoTokenizer.from_pretrained(id_modelo)
    modelo = AutoModelForSequenceClassification.from_pretrained(
        id_modelo, num_labels=len(EMOCOES) if multirrotulo else 3,
        problem_type='multi_label_classification' if multirrotulo else 'single_label_classification')

    def calcular(ep):
        logits, y = ep
        if multirrotulo:
            p = (1 / (1 + np.exp(-logits)) > HIPER['limiar_multirrotulo']).astype(int)
            return {'macro_f1': f1_score(y.astype(int), p, average='macro', zero_division=0)}
        return {'macro_f1': f1_score(y, logits.argmax(axis=1), average='macro', zero_division=0)}

    pasta = os.path.join(PASTA_TEMP, chave)
    args = TrainingArguments(
        output_dir=pasta, eval_strategy='epoch', save_strategy='epoch', save_total_limit=1, save_only_model=True,
        load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True,
        learning_rate=taxa, per_device_train_batch_size=HIPER['tamanho_lote'], per_device_eval_batch_size=64,
        num_train_epochs=HIPER['epocas'][tarefa], weight_decay=HIPER['decaimento_pesos'],
        warmup_ratio=HIPER['proporcao_aquecimento'], fp16=(DISPOSITIVO == 'cuda'), seed=semente, data_seed=semente,
        logging_strategy='epoch', report_to='none', disable_tqdm=not EM_COLAB, dataloader_num_workers=0)
    trainer = Trainer(model=modelo, args=args, train_dataset=preparar(tok, partes['train'], multirrotulo),
                      eval_dataset=preparar(tok, partes['dev'], multirrotulo), data_collator=DataCollatorWithPadding(tok),
                      compute_metrics=calcular, callbacks=[RegistoTreino(chave)])
    t0 = time.time()
    trainer.train()
    segundos = time.time() - t0
    te = partes['test']
    prev = prever(trainer, preparar(tok, te, multirrotulo), multirrotulo)
    m = metricas_emocoes(te[EMOCOES].values, prev) if multirrotulo else metricas_sentimento(te['y'], prev)
    registar_execucao(tarefa, nome_modelo, semente, m, te, prev, segundos, {'id_modelo': id_modelo, 'taxa_aprendizagem': taxa})
    if tarefa == 'senti_por':
        te_xi = DADOS['senti_tso']['test']
        prev_xi = prever(trainer, preparar(tok, te_xi, False), False)
        registar_execucao('zeroshot_por_tso', nome_modelo, semente, metricas_sentimento(te_xi['y'], prev_xi), te_xi, prev_xi,
                          extra={'id_modelo': id_modelo, 'treino': 'AfriSenti PT-MZ', 'teste': 'AfriSenti Xitsonga'})
    del trainer, modelo
    if DISPOSITIVO == 'cuda':
        torch.cuda.empty_cache()
    shutil.rmtree(pasta, ignore_errors=True)


t_inicio = time.time()
for nome_modelo, id_modelo in MODELOS.items():
    for tarefa in ['senti_por', 'senti_tso', 'emo_ptmz']:
        for semente in SEMENTES:
            ajustar(tarefa, nome_modelo, id_modelo, semente, HIPER['taxa_aprendizagem'])
log.info(f'[Treino] Modelos base concluídos em {(time.time() - t_inicio) / 60:.1f} min')

### Opcional — AfroXLMR-large-76L no AfriSenti Xitsonga
Este modelo inclui o xitsonga no pré-treino adaptativo. Se a GPU ficar sem memória, a execução é registada e saltada.

In [ ]:
if EXECUTAR_AFROXLMR_76L:
    for semente in SEMENTES:
        try:
            ajustar('senti_tso', MODELO_76L[0], MODELO_76L[1], semente, HIPER['taxa_aprendizagem_large'])
        except torch.cuda.OutOfMemoryError:
            log.warning(f'[76L] Memória da GPU insuficiente na semente {semente}; execução saltada.')
            torch.cuda.empty_cache()

## 8. Agregação dos resultados (média ± desvio-padrão das sementes)

In [ ]:
ORDEM_MODELOS = ['Maioritária', 'Léxico NRC', 'TF-IDF+RL'] + list(MODELOS) + [MODELO_76L[0]]
execucoes = pd.DataFrame([json.load(open(f, encoding='utf-8')) for f in glob.glob(os.path.join(SUB['execucoes'], '*.json'))])
execucoes.to_csv(os.path.join(SUB['tabelas'], 'T_resultados_todas_execucoes.csv'), index=False, encoding='utf-8-sig')


def media_dp(serie):
    return f'{serie.mean():.4f} ± {serie.std(ddof=1):.4f}' if len(serie) > 1 else f'{serie.mean():.4f}'


def resumir(tarefas, metricas):
    sub = execucoes[execucoes['tarefa'].isin(tarefas)]
    linhas = []
    for (tarefa, modelo), g in sub.groupby(['tarefa', 'modelo']):
        linha = {'tarefa': TAREFAS[tarefa], 'modelo': modelo, 'n_sementes': len(g)}
        for m in metricas:
            linha[m] = media_dp(g[m])
            linha[m + '_media'] = round(g[m].mean(), 4)
        linhas.append(linha)
    df = pd.DataFrame(linhas)
    df['ordem'] = df['modelo'].map({m: i for i, m in enumerate(ORDEM_MODELOS)})
    return df.sort_values(['tarefa', 'ordem']).drop(columns='ordem')


T_sent = resumir(['senti_por', 'senti_tso'], ['macro_f1', 'f1_ponderado', 'exactidao', 'kappa_cohen'])
T_zero = resumir(['zeroshot_por_tso'], ['macro_f1', 'f1_ponderado', 'exactidao'])
T_emo = resumir(['emo_ptmz'], ['macro_f1', 'micro_f1'] + [f'f1_{e}' for e in EMOCOES])
T_sent.to_csv(os.path.join(SUB['tabelas'], 'T_sentimento_resumo.csv'), index=False, encoding='utf-8-sig')
T_zero.to_csv(os.path.join(SUB['tabelas'], 'T_zero_shot_resumo.csv'), index=False, encoding='utf-8-sig')
T_emo.to_csv(os.path.join(SUB['tabelas'], 'T_emocoes_resumo.csv'), index=False, encoding='utf-8-sig')

tempos = (execucoes[execucoes['segundos_treino'] > 0].groupby(['tarefa', 'modelo'])['segundos_treino']
          .agg(['mean', 'sum', 'count']).round(1).reset_index())
tempos.to_csv(os.path.join(SUB['tabelas'], 'T_tempos_treino.csv'), index=False, encoding='utf-8-sig')

for nome, t in [('SENTIMENTO', T_sent), ('ZERO-SHOT', T_zero), ('EMOÇÕES', T_emo)]:
    log.info(f'\n===== {nome} =====\n' + t[[c for c in t.columns if not c.endswith('_media')]].to_string(index=False))

## 9. E5 — Desempenho por categoria linguística
O detector em dois estágios do protótipo (léxico xichangana + langdetect) é aplicado aos textos de teste do AfriSenti;
os rótulos humanos de sentimento mantêm-se e o Macro-F1 é calculado por categoria (média das sementes).

In [ ]:
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
DetectorFactory.seed = 42
with open(os.path.join(PASTA_DADOS, 'lexico_xichangana_set.txt'), encoding='utf-8') as f:
    LEXICO_XI = {l.strip().lower() for l in f if l.strip()}


def classificar_lingua(texto):
    toks = re.findall(r"\b[\w']+\b", texto.lower())
    score = sum(t in LEXICO_XI for t in toks) / len(toks) if toks else 0.0
    try:
        base = detect(texto)
    except LangDetectException:
        base = 'unknown'
    if score >= 0.8:
        return 'Xichangana'
    if score >= 0.3:
        return 'Código-Misto'
    if score < 0.05 and base == 'pt':
        return 'Português'
    return 'Código-Misto'


categorias = {t: DADOS[t]['test']['texto'].apply(classificar_lingua).values for t in ['senti_por', 'senti_tso']}
linhas = []
for tarefa in ['senti_por', 'senti_tso', 'zeroshot_por_tso']:
    cat = categorias['senti_tso' if tarefa == 'zeroshot_por_tso' else tarefa]
    for modelo in execucoes.loc[execucoes['tarefa'] == tarefa, 'modelo'].unique():
        for c in ['Português', 'Código-Misto', 'Xichangana']:
            mascara = cat == c
            if mascara.sum() == 0:
                continue
            valores = []
            for f in glob.glob(os.path.join(SUB['previsoes'], f'{tarefa}__{modelo}__s*.csv')):
                p = pd.read_csv(f)
                valores.append(f1_score(p.loc[mascara, 'real'], p.loc[mascara, 'previsto'], average='macro',
                                        labels=SENTIMENTOS, zero_division=0))
            linhas.append({'tarefa': TAREFAS[tarefa], 'modelo': modelo, 'categoria': c, 'n_textos': int(mascara.sum()),
                           'macro_f1': media_dp(pd.Series(valores)), 'macro_f1_media': round(np.mean(valores), 4),
                           'aviso': 'amostra reduzida (n < 30)' if mascara.sum() < 30 else ''})
T_cat = pd.DataFrame(linhas)
T_cat.to_csv(os.path.join(SUB['tabelas'], 'T_por_categoria_linguistica.csv'), index=False, encoding='utf-8-sig')
dist_cat = {t: pd.Series(c).value_counts().to_dict() for t, c in categorias.items()}
log.info('[Categorias] ' + json.dumps(dist_cat, ensure_ascii=False))

## 10. E6 — Figuras: matrizes de confusão e F1 por emoção

In [ ]:
plt.rcParams.update({'font.size': 10.5, 'axes.titleweight': 'bold', 'figure.facecolor': 'white'})
AZUL = '#2a78d6'


def virgula(x, casas=1):
    return f'{x:.{casas}f}'.replace('.', ',')


def melhor_modelo(tarefa):
    sub = execucoes[(execucoes['tarefa'] == tarefa) & (~execucoes['modelo'].isin(['Maioritária']))]
    return sub.groupby('modelo')['macro_f1'].mean().idxmax()


def matriz_confusao(tarefa, ax, titulo):
    modelo = melhor_modelo(tarefa)
    ficheiros = sorted(glob.glob(os.path.join(SUB['previsoes'], f'{tarefa}__{modelo}__s*.csv')))
    total = sum(confusion_matrix(pd.read_csv(f)['real'], pd.read_csv(f)['previsto'], labels=SENTIMENTOS) for f in ficheiros)
    media = total / len(ficheiros)
    pct = media / media.sum(axis=1, keepdims=True) * 100
    anot = np.array([[f'{virgula(pct[i, j])}%\n({virgula(media[i, j], 0 if len(ficheiros) == 1 else 1)})'
                      for j in range(3)] for i in range(3)])
    rot = [SENT_PT[s] for s in SENTIMENTOS]
    sns.heatmap(pct, annot=anot, fmt='', cmap='Blues', vmin=0, vmax=100, cbar=False, xticklabels=rot, yticklabels=rot,
                linewidths=2, linecolor='white', ax=ax, annot_kws={'fontsize': 9.5})
    ax.set_xlabel('Classe prevista')
    ax.set_ylabel('Classe real')
    ax.set_title(f'{titulo}\n{modelo}', fontsize=11)
    return modelo


fig, eixos = plt.subplots(1, 3, figsize=(14, 4.6))
melhores = {}
for ax, (tarefa, titulo) in zip(eixos, [('senti_por', 'Português MZ'), ('senti_tso', 'Xitsonga'),
                                        ('zeroshot_por_tso', 'Zero-shot PT-MZ → Xitsonga')]):
    melhores[tarefa] = matriz_confusao(tarefa, ax, titulo)
plt.tight_layout()
plt.savefig(os.path.join(SUB['figuras'], 'figura_matrizes_confusao_sentimento.png'), dpi=200, bbox_inches='tight')
plt.close()

melhores['emo_ptmz'] = melhor_modelo('emo_ptmz')
fig, eixos = plt.subplots(2, 3, figsize=(11, 6.6))
ficheiros = sorted(glob.glob(os.path.join(SUB['previsoes'], f'emo_ptmz__{melhores["emo_ptmz"]}__s*.csv')))
mcm = sum(multilabel_confusion_matrix(pd.read_csv(f)[[f'real_{e}' for e in EMOCOES]].values,
                                      pd.read_csv(f)[[f'prev_{e}' for e in EMOCOES]].values) for f in ficheiros) / len(ficheiros)
for ax, e, m in zip(eixos.ravel(), EMOCOES, mcm):
    sns.heatmap(m, annot=np.vectorize(lambda v: virgula(v, 0 if len(ficheiros) == 1 else 1))(m), fmt='', cmap='Blues',
                cbar=False, xticklabels=['Ausente', 'Presente'], yticklabels=['Ausente', 'Presente'],
                linewidths=2, linecolor='white', ax=ax)
    ax.set_title(EMO_PT[e], fontsize=11)
    ax.set_xlabel('Previsto')
    ax.set_ylabel('Real')
fig.suptitle(f'Matrizes de confusão por emoção — {melhores["emo_ptmz"]}', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SUB['figuras'], 'figura_matrizes_confusao_emocoes.png'), dpi=200, bbox_inches='tight')
plt.close()

modelos_emo = [m for m in ORDEM_MODELOS if m in T_emo['modelo'].values and m != 'Maioritária']
cores = ['#b9b8b2', '#52514e', '#eb6834', '#1baf7a', '#2a78d6', '#4a3aa7'][:len(modelos_emo)]
fig, ax = plt.subplots(figsize=(10, 4.8))
x = np.arange(len(EMOCOES))
larg = 0.8 / len(modelos_emo)
for i, (modelo, cor) in enumerate(zip(modelos_emo, cores)):
    linha = T_emo[T_emo['modelo'] == modelo].iloc[0]
    ax.bar(x + (i - (len(modelos_emo) - 1) / 2) * larg, [linha[f'f1_{e}_media'] * 100 for e in EMOCOES], larg * 0.92,
           color=cor, label=modelo)
ax.set_xticks(x)
ax.set_xticklabels([EMO_PT[e] for e in EMOCOES])
ax.set_ylabel('F1 (%)')
ax.set_ylim(0, 100)
ax.yaxis.grid(True, color='#e4e3df')
ax.set_axisbelow(True)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(ncol=len(modelos_emo), loc='lower center', bbox_to_anchor=(0.5, 1.0), frameon=False)
ax.set_title('F1 por emoção no AfriEmo — Português MZ', pad=30)
plt.tight_layout()
plt.savefig(os.path.join(SUB['figuras'], 'grafico_f1_por_emocao.png'), dpi=200, bbox_inches='tight')
plt.close()

sent_modelos = [m for m in ORDEM_MODELOS if m in T_sent['modelo'].values and m != 'Maioritária']
fig, ax = plt.subplots(figsize=(10, 4.8))
conj = [('Sentimento — AfriSenti Português MZ', 'Português MZ'), ('Sentimento — AfriSenti Xitsonga', 'Xitsonga'),
        (TAREFAS['zeroshot_por_tso'], 'Zero-shot PT-MZ → XI')]
cores_m = dict(zip(['Léxico NRC', 'TF-IDF+RL'] + list(MODELOS) + [MODELO_76L[0]],
                   ['#b9b8b2', '#52514e', '#eb6834', '#1baf7a', '#2a78d6', '#4a3aa7']))
todos = pd.concat([T_sent, T_zero])
x = np.arange(len(conj))
larg = 0.8 / len(sent_modelos)
for i, modelo in enumerate(sent_modelos):
    vals, erros = [], []
    for nome_t, _ in conj:
        g = execucoes[(execucoes['tarefa'] == {v: k for k, v in TAREFAS.items()}[nome_t]) & (execucoes['modelo'] == modelo)]['macro_f1']
        vals.append(g.mean() * 100 if len(g) else np.nan)
        erros.append(g.std(ddof=1) * 100 if len(g) > 1 else np.nan)
    ax.bar(x + (i - (len(sent_modelos) - 1) / 2) * larg, vals, larg * 0.92, yerr=erros, capsize=3,
           color=cores_m[modelo], label=modelo, error_kw={'elinewidth': 1, 'ecolor': '#0b0b0b'})
ax.set_xticks(x)
ax.set_xticklabels([c[1] for c in conj])
ax.set_ylabel('Macro-F1 (%)')
ax.set_ylim(0, 100)
ax.yaxis.grid(True, color='#e4e3df')
ax.set_axisbelow(True)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(ncol=min(len(sent_modelos), 6), loc='lower center', bbox_to_anchor=(0.5, 1.0), frameon=False)
ax.set_title('Macro-F1 na classificação de sentimento por modelo', pad=30)
plt.tight_layout()
plt.savefig(os.path.join(SUB['figuras'], 'grafico_macro_f1_sentimento.png'), dpi=200, bbox_inches='tight')
plt.close()
log.info('[Figuras] ' + ', '.join(sorted(os.listdir(SUB['figuras']))))

## 11. Material para os anexos e exportação

In [ ]:
config = {'hiperparametros': HIPER, 'sementes': SEMENTES, 'modelos': MODELOS,
          'modelo_opcional': dict([MODELO_76L]) if EXECUTAR_AFROXLMR_76L else {},
          'limpeza_texto': 'URLs, menções (@utilizador), HTML e caracteres de controlo (regras do cleaner.py)',
          'baselines': {'Maioritária': 'classe mais frequente no treino',
                        'TF-IDF+RL': 'TF-IDF de palavras (1-2) e caracteres (2-5) + Regressão Logística com pesos balanceados',
                        'Léxico NRC': 'NRC EmoLex v0.92, tradução portuguesa; positivo>negativo → Positivo, inverso → Negativo, empate → Neutro'},
          'metricas': {'sentimento': 'Macro-F1 (principal), F1 ponderado (comparação com AfriSenti), exactidão, Kappa de Cohen',
                       'emocoes': 'Macro-F1 sobre as 6 emoções (métrica oficial da SemEval-2025 Task 11, faixa A), Micro-F1'},
          'ambiente': ambiente}
with open(os.path.join(SUB['anexos'], 'Anexo_configuracao_experimental.json'), 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

with open(os.path.join(SUB['anexos'], 'Anexo_relatorios_classificacao.txt'), 'w', encoding='utf-8') as f:
    for tarefa in ['senti_por', 'senti_tso', 'zeroshot_por_tso', 'emo_ptmz']:
        for modelo in [m for m in ORDEM_MODELOS if m in execucoes.loc[execucoes['tarefa'] == tarefa, 'modelo'].values]:
            sem = semente_de(modelo)
            caminho = os.path.join(SUB['relatorios'], chave_execucao(tarefa, modelo, sem) + '.txt')
            if os.path.exists(caminho):
                f.write(open(caminho, encoding='utf-8').read() + '\n' + '-' * 70 + '\n')

exemplos = []
for tarefa in ['senti_por', 'senti_tso']:
    p = pd.read_csv(os.path.join(SUB['previsoes'], chave_execucao(tarefa, melhores[tarefa], semente_de(melhores[tarefa])) + '.csv'))
    for acerto, g in [(True, p[p['real'] == p['previsto']]), (False, p[p['real'] != p['previsto']])]:
        for _, r in g.sample(min(8, len(g)), random_state=42).iterrows():
            exemplos.append({'tarefa': TAREFAS[tarefa], 'modelo': melhores[tarefa], 'texto': r['texto'],
                             'real': SENT_PT[r['real']], 'previsto': SENT_PT[r['previsto']], 'correcto': 'Sim' if acerto else 'Não'})
p = pd.read_csv(os.path.join(SUB['previsoes'], chave_execucao('emo_ptmz', melhores['emo_ptmz'], semente_de(melhores['emo_ptmz'])) + '.csv'))
reais = p[[f'real_{e}' for e in EMOCOES]].values
prevs = p[[f'prev_{e}' for e in EMOCOES]].values
iguais = (reais == prevs).all(axis=1)
nomes = lambda v: ', '.join(EMO_PT[e] for e, b in zip(EMOCOES, v) if b) or 'Neutro'
for acerto in [True, False]:
    idx = np.where(iguais == acerto)[0]
    for i in pd.Series(idx).sample(min(8, len(idx)), random_state=42):
        exemplos.append({'tarefa': TAREFAS['emo_ptmz'], 'modelo': melhores['emo_ptmz'], 'texto': p.loc[i, 'texto'],
                         'real': nomes(reais[i]), 'previsto': nomes(prevs[i]), 'correcto': 'Sim' if acerto else 'Não'})
pd.DataFrame(exemplos).to_csv(os.path.join(SUB['anexos'], 'Anexo_exemplos_previsoes.csv'), index=False, encoding='utf-8-sig')

log.info('[Fim] Avaliação concluída. Melhores modelos: ' + json.dumps(melhores, ensure_ascii=False))
for h in log.handlers:
    h.flush()
with open(os.path.join(SUB['anexos'], 'Anexo_log_execucao.log'), 'w', encoding='utf-8') as destino:
    for caminho in sorted(glob.glob(os.path.join(SUB['logs'], 'execucao_*.log'))):
        destino.write(open(caminho, encoding='utf-8').read())

zip_destino = os.path.join(PASTA_BASE, 'resultados_EmoText_teste' if TESTE_RAPIDO else 'resultados_EmoText')
shutil.make_archive(zip_destino, 'zip', PASTA_RES)
print('Ficheiro de resultados:', zip_destino + '.zip')
if EM_COLAB:
    from google.colab import files
    files.download(zip_destino + '.zip')